# Traditional ML Biometric Verification
One notebook, plain functions, and one visible pipeline at a time. Open the separate **02_Streamlit_Demo.ipynb** for the application.

**Educational virtual identities:** the supplied face, public speaker, and public finger assigned to one user belong to unrelated physical people. This is not genuine same-person multimodal authentication. No deep learning, pretrained recognition embeddings, or speech-to-text service is used.

**Data path:** real files -> fixed train/validation/test splits -> traditional features -> training-only StandardScaler + PCA -> KNN or linear SVM.

**Protected path:** claimed identity -> face -> speaker -> face + voice feature fusion -> fingerprint -> three-modality feature fusion -> command -> permissions -> simulated action. Every failed gate stops the path.

Install requirements into the selected Python environment before Run All. The Python helpers are shared with the UI so the notebook and app cannot silently use different algorithms. The published checkout uses the included demo samples and saved models. Add the full prepared dataset locally to run the training example and recompute held-out metrics.

In [ ]:
import sys
import json
import inspect
import subprocess
import numpy as np
import pandas as pd
from IPython.display import display, Code, Audio, Image as DisplayImage
from config import FACE_ROOT, LIVE, MANIFESTS, MODELS, ROOT
print('Python:', sys.executable)
print('Project:', ROOT)

## 1. Discover the Three People
**Input:** the original datasets/images folder. **Operation:** discover names, sort deterministically, assign user_01 to user_03. **Output:** the actual names and image counts. No original file is modified. In the published demo, use the saved identity mapping.

In [ ]:
from dataset_setup import discover_people, setup, validate_manifest

if FACE_ROOT.is_dir():
    people = discover_people()
    display(pd.DataFrame([{'user_id': user, 'display_name': folder.name,
        'image_files': len([path for path in folder.iterdir()
                            if path.suffix.lower() in {'.png', '.jpg', '.jpeg'}])}
        for user, folder in people.items()]))
else:
    print('Using the saved identity mapping below.')

## 2. Prepare Real Data and Freeze the Splits
**Input:** supplied images, Speech Commands v0.02, and official FVC2004 DB1_B. **Operation:** decode, reject corrupt/duplicate content, warn about similar photos, select one real speaker and finger per person, and split before training. **Output:** sample, virtual-identity, pair, triplet, and held-out demo manifests.

For 40 faces per person the split is 24/8/8. Each speaker has five distinct utterances for each of ON, OFF, GO, STOP, YES, NO: 3/1/1 per word. YES and NO become UNKNOWN for the command classifier. Fingerprints use 5/1/2 impressions per finger.

Speech uses local utterance-level partitions of enrolled speakers, not official Speech Commands benchmark partitions. A demo copy remains the same test observation, not an extra sample. Source decoding/hashes cannot prove separate photo capture sessions. Similar images can inflate results.

Leave REBUILD_DATA=False to preserve the prepared experiment. Set it to True only after supplying the authorized original files. If downloading fails, the script reports BLOCKED; never substitute fake samples. Manual recovery is documented in DATASET_SOURCES.md.

In [ ]:
REBUILD_DATA = False
if REBUILD_DATA:
    setup(download=True)

samples = pd.read_csv(MANIFESTS / 'samples.csv', keep_default_na=False)
identities = json.loads((MANIFESTS / 'identities.json').read_text(encoding='utf-8'))
has_prepared_data = not samples.empty and all((ROOT / path).is_file() for path in samples['path'])
if has_prepared_data:
    assert validate_manifest(samples)
    quality_path = MANIFESTS / 'face_quality.json'
    if quality_path.is_file():
        quality = json.loads(quality_path.read_text(encoding='utf-8'))
        print('Near-duplicate warnings:', {row['person']: len(row['near_duplicate_pairs'])
                                          for row in quality['warnings']})
        print('Corrupt/exact-duplicate exclusions:', quality['rejected'])
else:
    print('Demo mode: showing the recorded splits and using the included samples for inference.')
display(samples.groupby(['modality', 'user_id', 'split']).size().unstack(fill_value=0))
display(pd.DataFrame(identities).T)

demo_manifest = LIVE / 'manifest.csv'
if has_prepared_data and demo_manifest.is_file():
    demo = pd.read_csv(demo_manifest)
else:
    demo_rows = []
    for user_id, identity in identities.items():
        name = identity['display_name']
        folder = ROOT / 'datasets' / 'demo' / name
        demo_rows.append({'user_id': user_id, 'display_name': name,
                          'face_path': folder / 'face.jpg', 'voice_path': folder / 'voice.wav',
                          'fingerprint_path': folder / 'fingerprint.png'})
    demo = pd.DataFrame(demo_rows)
display(demo[['user_id', 'display_name']])

def feature_example(modality):
    if has_prepared_data:
        return ROOT / samples[samples.modality.eq(modality)].iloc[0]['path']
    return ROOT / demo.iloc[0][f'{modality}_path']

## 3. Face Feature Pipeline
**Input:** RGB image. **Operation:** center-fit to 128 x 128; grayscale copy; HOG describes edges; normalized uniform LBP describes texture; three 256-bin histograms describe color. **Output:** one concatenated numerical vector.

HOG uses 9 orientations, 8 x 8 cells, and 2 x 2 blocks. LBP uses 8 neighbors, radius 1. The actual length is printed, not assumed. The displayed source is the same function used in the UI.

In [ ]:
from features import load_image, hog_lbp, face_features, voice_features, fingerprint_features
display(Code(inspect.getsource(hog_lbp), language='python'))
display(Code(inspect.getsource(face_features), language='python'))
face_path = feature_example('face')
face_raw = face_features(face_path)
print('Face features:', face_raw.shape)
display(DisplayImage(filename=str(face_path), width=220))

## 4. Voice Feature Pipeline
**Input:** a real isolated-word recording. **Operation:** mono, 16 kHz, MFCC (13 coefficients), zero-crossing rate (ZCR), and RMS energy; mean and standard deviation across frames. **Output:** 13 + 13 + 1 + 1 + 1 + 1 = **30 values**. Empty, silent, corrupt, or excessively long audio is rejected.

Speaker identity and spoken word are different targets. Both classifiers use exactly the same extracted features; this is isolated-word classification, not general speech-to-text.

In [ ]:
display(Code(inspect.getsource(voice_features), language='python'))
audio_path = feature_example('voice')
voice_raw = voice_features(audio_path)
assert voice_raw.shape == (30,)
print('Voice features:', voice_raw.shape)
display(Audio(filename=str(audio_path)))

## 5. Fingerprint Feature Pipeline
**Input:** a genuine DB1_B impression. **Operation:** grayscale, center-fit/resize, HOG + normalized LBP. **Output:** one feature vector. Each virtual identity uses eight distinct impressions of one original finger. DB4_B is synthetic and is not used. No scanner or advanced minutiae algorithm is required.

In [ ]:
display(Code(inspect.getsource(fingerprint_features), language='python'))
finger_path = feature_example('fingerprint')
print('Fingerprint features:', fingerprint_features(finger_path).shape)
display(DisplayImage(filename=str(finger_path), width=180))

## 6. StandardScaler and PCA: Fit Only Training Data
**Input:** already separated raw feature vectors. **Operation:** fit StandardScaler and PCA on training rows only; transform validation/test with those fixed transforms. **Output:** smaller vectors preserving at least approximately 95% of training variance.

PCA does not use class labels, but fitting it on test data still leaks the test distribution. Split FIRST, fit SECOND. PCA whitening puts retained components on comparable training variance. Each modality block is divided by sqrt(number of retained features), so a longer face vector does not dominate voice purely by length. The same saved transformations are used in live inference.

In [ ]:
from train import prepare_feature_data, task_data, train_models
display(Code(inspect.getsource(prepare_feature_data), language='python'))
if has_prepared_data:
    preprocessors, vectors, dimensions = prepare_feature_data(samples)
    tasks = task_data(samples, vectors)
    display(pd.DataFrame([{'task': task, 'train': len(splits['train'][1]),
        'validation': len(splits['validation'][1]), 'test': len(splits['test'][1]),
        'features': splits['train'][0].shape[1]} for task, splits in tasks.items()]))
else:
    print('Saved preprocessing dimensions')
    dimensions = json.loads((MODELS / 'model_index.json').read_text(encoding='utf-8'))['dimensions']
display(pd.DataFrame(dimensions).T)

## 7. Two Simple Models for Every Task
**Input:** training and validation feature matrices and target labels. **Operation:** KNeighborsClassifier(n_neighbors=3) and SVC(kernel='linear'); compare validation predictions. **Output:** twelve actual saved classifiers: face, speaker, fingerprint, face+voice, three modalities, and command, each with KNN and SVM.

The short loop below shows the exact fit/predict process for all six tasks. Saved models and distance thresholds come from train_models(). Set RETRAIN=True only for an intentional new training run. Selection uses validation macro-F1, then validation accuracy, then KNN on exact ties. Test results never choose the winner.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from verification import load_system

RETRAIN = False
if RETRAIN:
    if not has_prepared_data:
        raise ValueError('Prepare the full datasets before retraining.')
    train_models()

if has_prepared_data:
    for task, splits in tasks.items():
        training, training_labels, _ = splits['train']
        validation, validation_labels, _ = splits['validation']
        for name, model in [('KNN', KNeighborsClassifier(n_neighbors=3)),
                            ('SVM', SVC(kernel='linear'))]:
            model.fit(training, training_labels)
            predicted = model.predict(validation)
            print(task, name, 'validation accuracy:', round(accuracy_score(validation_labels, predicted), 3))
else:
    print('Using the saved classifiers and recorded validation scores.')
system = load_system()
print('Selected using validation only:', system['selected'])
display(pd.read_csv(ROOT / 'results/metrics/model_comparison.csv'))

## 8. Identification Is Not Verification
**Identification:** which enrolled class is predicted? **Verification:** does this sample support the user's particular claim?

**Input:** a claimed user and a transformed vector. **Operation:** predict; measure Euclidean distance to that claimed user's training centroid; accept only if the prediction matches AND the distance is within a validation-calibrated threshold. **Output:** accepted, claimed_identity, predicted_identity, distance, threshold, explanation.

For each validation vector, try one genuine and both impostor claims. Threshold selection minimizes FAR + FRR; ties prefer lower FAR and then the smaller threshold. This basic rule is an educational approximation, not liveness detection or certified unknown-person rejection.

In [ ]:
from verification import calibrate_threshold, verify_vector, verify_face, verify_voice
display(Code(inspect.getsource(calibrate_threshold), language='python'))
display(Code(inspect.getsource(verify_vector), language='python'))
chosen = demo.iloc[0]
claimed = chosen['user_id']
face = ROOT / chosen['face_path']
voice = ROOT / chosen['voice_path']
finger = ROOT / chosen['fingerprint_path']
print('Prepared person / claim:', chosen['display_name'], claimed)
print('Face-only result:')
display(verify_face(face, claimed, system))
print('Speaker-only result:')
display(verify_voice(voice, claimed, system))

## 9. Face + Voice Early Feature Fusion
**Input:** separate transformed face and voice vectors from the same virtual user and split. **Operation:** concatenate numerical values; train a dedicated fusion classifier, not an AND of class predictions. **Output:** a genuinely trained fusion verifier.

The pair manifest does not repeat either source sample. Public voices do not physically belong to the people in the photos: this pairing is explicitly virtual.

In [ ]:
from verification import extract_vector, verify_face_voice_fusion
from fusion import concatenate_features
face_vector = extract_vector('face', face, system)
voice_vector = extract_vector('voice', voice, system)
fusion_vector = np.concatenate([face_vector, voice_vector])
print('Face:', face_vector.shape, 'Voice:', voice_vector.shape, 'Fusion:', fusion_vector.shape)
display(verify_face_voice_fusion(face, voice, claimed, system))
display(pd.read_csv(MANIFESTS / 'face_voice_pairs.csv').groupby(['user_id', 'split']).size().unstack())

## 10. Fingerprint Verification and Three-Modality Fusion
**Input:** one fingerprint impression and the face/voice blocks. **Operation:** verify the fingerprint independently, concatenate three blocks, then run a separately trained fusion3 model. **Output:** both actual decisions.

There are only five training, one validation, and two test triplets per person, each using a distinct fingerprint. We do not multiply combinations and call them independent observations. A rejection is retained, not repaired by changing the held-out file or threshold.

In [ ]:
from verification import verify_fingerprint, verify_three_modality_fusion
display(verify_fingerprint(finger, claimed, system))
fingerprint_vector = extract_vector('fingerprint', finger, system)
fusion3_vector = np.concatenate([face_vector, voice_vector, fingerprint_vector])
print('Three-modality feature shape:', fusion3_vector.shape)
display(verify_three_modality_fusion(face, voice, finger, claimed, system))
display(pd.read_csv(MANIFESTS / 'three_modality_triplets.csv').groupby(['user_id', 'split']).size().unstack())

## 11. Command Recognition and Permissions Are Different
The word classifier predicts ON, OFF, GO, STOP, or UNKNOWN using real audio. Unsupported YES/NO training recordings provide UNKNOWN examples. A validation distance check can additionally reject distant audio. A rejected word never executes. An incorrect word prediction is still possible and must be measured.

**Authorization input:** verified_user_id and recognized_command. **Operation:** a dictionary lookup, not ML. **Output:** ALLOWED or DENIED. Admin gets all four commands; Employee gets OFF/GO/STOP; Guest gets OFF/STOP. These dictionary examples alone are not biometric authentication results.

In [ ]:
from permissions import PERMISSIONS, check_permission
from commands import recognize_command
display(PERMISSIONS)
display(Code(inspect.getsource(recognize_command), language='python'))
print('Admin + ON:', check_permission('user_01', 'ON'))
print('Employee + ON:', check_permission('user_02', 'ON'))
print('Guest + GO:', check_permission('user_03', 'GO'))
print('Guest + STOP:', check_permission('user_03', 'STOP'))

## 12. The Complete Hierarchical Pipeline
**Input:** three files and one claim. **Operation:** face -> speaker -> face+voice -> fingerprint -> fusion3 -> word -> permission -> simulated action. **Output:** a full decision trace.

A failure immediately returns; later gates are NOT RUN. The voice vector is extracted once and reused for speaker, both fusion stages, and command recognition. Recognition is not called until all biometric gates pass. No model is retrained at a click. This notebook exercises the complete research pipeline. The Streamlit sign-in uses the individual face, voice, and fingerprint checks.

In [ ]:
from pipeline import run_pipeline
display(Code(inspect.getsource(run_pipeline), language='python'))
result = run_pipeline(face, voice, finger, claimed, system)
display(pd.DataFrame(result['trace']))
print('Final:', result['result'])
print('Reason:', result['explanation'])
print('Recognized command:', result['command'])
print('Executed action:', result['action'])

## 13. Genuine and Impostor Claims on Actual Models
**Input:** the three published demo folders or the original held-out triplets when available locally. **Operation:** run each folder against every enrolled claim. **Output:** actual decisions, including genuine failures and any false accepts. Wrong claims are not forced to fail. The nine claims reuse three source triplets, so they are not nine independent biometric observations.

In [ ]:
outputs = []
for row in demo.to_dict('records'):
    for claim in identities:
        decision = run_pipeline(ROOT / row['face_path'], ROOT / row['voice_path'],
                                ROOT / row['fingerprint_path'], claim, system)
        outputs.append({'source': row['display_name'], 'claim': identities[claim]['display_name'],
                        'result': decision['result'], 'authenticated': decision['authenticated'],
                        'action': decision['action']})
display(pd.DataFrame(outputs))

## 14. Evaluate Frozen Models on Test Data
**Input:** held-out features, source manifests, and frozen saved models. **Operation:** evaluate both algorithms on six tasks, five verifiers, and the complete hierarchy. **Output:** real accuracy, precision, recall, F1, confusion matrices, FAR and FRR with their denominators.

FAR = false accepts / impostor attempts. FRR = false rejects / genuine attempts. Command FAR instead means unsupported words accepted as commands; do not confuse it with identity FAR. A model can identify correctly but still reject at the distance gate. An action can satisfy permissions for the predicted word but be unintended because that word was recognized incorrectly.

No threshold, speaker, demo file or model is selected using these test results. Zero observed FAR on a tiny enrolled-person test is not a security guarantee. Test claims sharing one source observation are dependent. Similar portraits also limit generalization.

In [ ]:
from evaluate import evaluate

if has_prepared_data:
    evaluation_report = evaluate()
else:
    print('Showing the recorded held-out evaluation. Recomputing it requires the full prepared dataset.')
metrics = pd.read_csv(ROOT / 'results/metrics/test_metrics.csv')
display(metrics)
display(pd.read_csv(ROOT / 'results/metrics/hierarchy_metrics.csv'))
display(DisplayImage(filename=str(ROOT / 'results/confusion_matrices/command_knn_with_rejection.png'), width=550))

## 15. Run the Automated Tests
**Input:** the real modules, manifests, models and held-out samples. **Operation:** pytest checks routing, role permissions, every failure gate, UNKNOWN handling, sample separation, source consistency, all twelve classifiers and real fusion dimensions. **Output:** the actual test-run report.

Test doubles are used only for isolated control-flow cases. They are never presented as trained predictions. Data/model integration tests use actual files. A skipped integration test means unavailable evidence, not success.

In [ ]:
completed = subprocess.run([sys.executable, '-m', 'pytest', '-q'], cwd=ROOT,
                           text=True, capture_output=True)
print(completed.stdout)
if completed.returncode:
    print(completed.stderr)
assert completed.returncode == 0

## Next: The Local Application
Open **02_Streamlit_Demo.ipynb** or run `python -m streamlit run app.py` from this folder. The app uses the saved camera model to identify the person, then verifies the voice and fingerprint one step at a time.

Webcam mode means **Live Face + Offline Voice + Offline Fingerprint**. It requires a genuine new camera snapshot; a saved image is never substituted. Implementation does not prove a live camera demonstration.

Read **README.md** for commands and actual results; **DATASET_SOURCES.md** for provenance and manual recovery; **ORAL_DEFENSE.md** for English and Egyptian Arabic explanations.

**Limitations:** tiny virtual-identity experiment; near-duplicate portraits; only three fingerprint validation impressions; imperfect command/UNKNOWN recognition; no liveness, spoof resistance, real system control or production security claim. Team-collected voice data is **NOT SATISFIED**, since public recordings were requested. Preserve poor results rather than manufacturing a successful demonstration.